In [ ]:
import pandas as pd
import pyreadstat
import re
import os

RAW_DIR=r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data"
OUT_PATH=os.path.join(RAW_DIR,"individual_level_london.csv")

sav_files={"surveydata1617.sav": "2016-17","surveydata1718.sav": "2017-18","surveydata1819.sav": "2018-19","surveydata1920.sav": "2019-20","surveydata2021.sav": "2020-21", "surveydata2122.sav": "2021-22","surveydata2223.sav": "2022-23",}

likert_map={"Strongly disagree": 1,"Disagree": 2,"Neither agree nor disagree": 3,"Agree": 4,"Strongly agree": 5,}

london_boroughs=["Barking and Dagenham","Barnet","Bexley","Brent","Bromley","Camden","City of London","Croydon","Ealing","Enfield","Greenwich","Hackney","Hammersmith and Fulham","Haringey","Harrow","Havering","Hillingdon",
    "Hounslow","Islington","Kensington and Chelsea","Kingston upon Thames","Lambeth","Lewisham","Merton","Newham","Redbridge","Richmond upon Thames","Southwark","Sutton","Tower Hamlets","Waltham Forest","Wandsworth","Westminster"]

In [ ]:
def load_and_clean_year(filename,year_label):
    path=os.path.join(RAW_DIR,filename)
    _,meta=pyreadstat.read_sav(path,metadataonly=True)
    available_cols=meta.column_names

    wanted=["LA","LA_2023","Age9","Gend3","Disab3","NSSEC5","wt_final","READYOP1_POP","READYOP_CV_3_POP","READYAB1_POP","MEMS7GR_ALL","serial"]
    usecols=[c for c in wanted if c in available_cols]
    df,meta=pyreadstat.read_sav(path,usecols=usecols,apply_value_formats=True)
    imd_raw,_=pyreadstat.read_sav(path,usecols=['serial','IMD10'],apply_value_formats=False)
    imd_raw=imd_raw.rename(columns={'IMD10': 'imd_decile'})
    df=df.merge(imd_raw,on='serial',how='left')

    borough_col="LA" if "LA" in df.columns else "LA_2023"
    df[borough_col]=df[borough_col].astype(str).str.replace(r'^E\d{8}\s+','',regex=True).str.strip()
    df=df[df[borough_col].isin(london_boroughs)].copy()
    df=df.rename(columns={borough_col: "borough"})

    #readiness under different variavle name
    readyop_col="READYOP1_POP" if "READYOP1_POP" in df.columns else "READYOP_CV_3_POP"
    readyab_col="READYAB1_POP"
    for col in [readyop_col,readyab_col]:
        if col in df.columns:
            df[col]=df[col].astype(str).str.strip().map(likert_map)
    df=df.rename(columns={readyop_col: "readiness_opportunity",readyab_col: "readiness_ability"})

    mems_str=df["MEMS7GR_ALL"].astype(str).str.strip()
    df["inactive"]=(mems_str == "Inactive").astype(int)
    keep_cols={"borough": "borough","Age9": "age_band","Gend3": "gend3","imd_decile": "imd_decile","Disab3": "disab3","NSSEC5": "nssec5","wt_final": "wt_final","readiness_opportunity": "readiness_opportunity","readiness_ability": "readiness_ability","inactive": "inactive",}
    available={k: v for k,v in keep_cols.items() if k in df.columns}
    df_out=df[list(available.keys())].rename(columns=available)

    df_out["survey_year"]=year_label
    df_out["covid_affected"]=year_label in ["2019-20","2020-21","2021-22"]
    return df_out

In [ ]:
all_years=[]
for fname,year in sav_files.items():
    print(f" {fname} ({year})")
    try:
        year_df=load_and_clean_year(fname,year)
        print(f" {len(year_df)} london respondents, " f"inactive rate: {year_df['inactive'].mean():.3f}")
        all_years.append(year_df)
    except Exception as e:
        print(f"  error {e}")

individual_df=pd.concat(all_years,ignore_index=True)
print(f"\n total individual-level rows: {len(individual_df)}")
gap_equity=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_equity.csv")
individual_df=individual_df.merge(gap_equity[["borough","s_bin"]],on="borough",how="left")

print("\n missing values per column:")
print(individual_df.isnull().sum())
print("\n inactive rate overall:",individual_df["inactive"].mean().round(3))
print("\nimd_decile values:",sorted(individual_df["imd_decile"].dropna().unique()))
print("\nRows per year:")
print(individual_df["survey_year"].value_counts().sort_index())

individual_df.to_csv(OUT_PATH,index=False)
print(f"\nSaved to {OUT_PATH}")
print(f"Shape: {individual_df.shape}")

In [ ]:
import statsmodels.formula.api as smf

df=pd.read_csv(OUT_PATH)
df['inactive']=df['inactive'].astype(int)
df['imd_decile']=pd.to_numeric(df['imd_decile'],errors='coerce')

cat_cols=['age_band','disab3','nssec5']
for c in cat_cols:
    if c in df.columns:
        df[c]=df[c].astype('category')

model_df=df.dropna(subset=['inactive','borough','imd_decile'] + cat_cols).copy()
print("Rows going into model:",len(model_df))
model_df['age_band_collapsed']=model_df['age_band'].astype(str).replace({'85+': '75+','75-84': '75+'})
model_df['age_band_collapsed']=model_df['age_band_collapsed'].astype('category')

print("\nage band counts:")
print(model_df['age_band_collapsed'].value_counts())
print("\nInactive rate by age band:")
print(model_df.groupby('age_band_collapsed',observed=True)['inactive'].mean())

print("\ninactive rate by borough:")
print(model_df.groupby('borough')['inactive'].mean().sort_values())

lpm_model=smf.mixedlm("inactive ~ C(age_band_collapsed) + imd_decile + C(disab3) + C(nssec5)",data=model_df,groups=model_df["borough"])
lpm_result=lpm_model.fit(method='powell',maxiter=200)
print(lpm_result.summary())

borough_var=lpm_result.cov_re.iloc[0,0]
resid_var=lpm_result.scale
icc=borough_var / (borough_var + resid_var)
print(f"\nBorough-level variance: {borough_var:.5f}")
print(f"Residual (individual) variance: {resid_var:.5f}")
print(f"ICC: {icc:.4f}  ->  {icc*100:.2f}% of variation is contributed to borough")

In [ ]:
model_df_int=model_df.dropna(subset=['s_bin']).copy()
model_df_int['s_bin']=model_df_int['s_bin'].astype('category')
model_df_fixed=model_df_int[model_df_int['nssec5'] != 'Aged <16 or 75+'].copy()
model_df_fixed['nssec5']=model_df_fixed['nssec5'].astype(str).astype('category')
model_df_fixed['age_band_collapsed']=model_df_fixed['age_band_collapsed'].astype(str).astype('category')

print(f"Rows before exclusion: {len(model_df_int)}")
print(f"Rows after excluding nssec5 == 'Aged <16 or 75+': {len(model_df_fixed)}")
print(f"Rows dropped: {len(model_df_int) - len(model_df_fixed)}")

interaction_model=smf.mixedlm("inactive ~ imd_decile * C(s_bin) + C(age_band_collapsed) + C(disab3) + C(nssec5)",data=model_df_fixed,groups=model_df_fixed["borough"])
interaction_result=interaction_model.fit(method='powell',maxiter=200)
print(interaction_result.summary())

In [ ]:
print(model_df_int.groupby(['nssec5','s_bin'],observed=True).size())

In [ ]:
try:
    borough_effects=pd.Series(lpm_result.random_effects).apply(lambda x: x.iloc[0])
    borough_effects=borough_effects.sort_values(ascending=False)
    print("\nHighest borough effects (worse than expected")
    print(borough_effects.head(10))
    print("\nLowest borough effects (better than expected)")
    print(borough_effects.tail(10))

    borough_effects.to_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\borough_random_effects.csv")
except Exception as e:
    print(f"problem {e}")

In [ ]:
import geopandas as gpd
from libpysal.weights import Queen
from esda.moran import Moran

gdf=gpd.read_file(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\london_boundaries_clean.geojson")
gap_equity=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_equity.csv")
print("Geojson columns:",gdf.columns.tolist())
print("Gap equity columns:",gap_equity.columns.tolist())

In [ ]:
import geopandas as gpd
from libpysal.weights import Queen
from esda.moran import Moran
from esda.moran import Moran_Local

gdf=gpd.read_file(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\london_boundaries_clean.geojson")
gap_equity=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_equity.csv")

gdf=gdf.merge(gap_equity,left_on="LAD24NM",right_on="borough",how="inner")
print("shape:",gdf.shape)
w=Queen.from_dataframe(gdf,use_index=False)
w.transform='r'
moran_demand=Moran(gdf['demand_score'].values,w)
print(f"\nMoran's I (demand_score): {moran_demand.I:.4f}, p-value: {moran_demand.p_sim:.4f}")
moran_equity=Moran(gdf['equity_gap'].values,w)
print(f"Moran's I (equity_gap): {moran_equity.I:.4f}, p-value: {moran_equity.p_sim:.4f}")

In [ ]:
lisa=Moran_Local(gdf['equity_gap'].values,w)
gdf['lisa_quadrant']=lisa.q
gdf['lisa_significant']=lisa.p_sim < 0.05
quadrant_labels={1: 'High-High cluster',2: 'Low-High outlier',3: 'Low-Low cluster',4: 'High-Low outlier'}
gdf['lisa_label']=gdf['lisa_quadrant'].map(quadrant_labels)
print(gdf[gdf['lisa_significant']][['borough','equity_gap','lisa_label']].sort_values('equity_gap',ascending=False))

gdf[['borough','equity_gap','lisa_label','lisa_significant']].to_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\lisa_equity_clusters.csv",index=False)

In [ ]:
venues=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\OpenActive_Merged.csv")
print(venues.columns.tolist())
print(venues.shape)

In [ ]:
venues=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\OpenActive_Merged.csv")
venues_geo=venues.dropna(subset=['latitude','longitude']).copy()
print(f"Venues with valid coordinates: {len(venues_geo)} / {len(venues)}")

venues_gdf=gpd.GeoDataFrame(venues_geo,geometry=gpd.points_from_xy(venues_geo['longitude'],venues_geo['latitude']),crs="EPSG:4326")
venues_gdf=venues_gdf.to_crs(epsg=27700)
gdf_bng=gdf.to_crs(epsg=27700)

gdf_bng['centroid']=gdf_bng.geometry.centroid
buffer_distances={'sessions_catchment_2km': 2000,'sessions_catchment_5km': 5000}
for col_name,dist in buffer_distances.items():
    counts=[]
    for centroid in gdf_bng['centroid']:
        buffer=centroid.buffer(dist)
        count=venues_gdf[venues_gdf.geometry.within(buffer)].shape[0]
        counts.append(count)
    gdf_bng[col_name]=counts

print(gdf_bng[['borough','sessions','sessions_catchment_2km','sessions_catchment_5km']].sort_values('sessions_catchment_5km',ascending=False))

In [ ]:
gdf_bng['s_bin_catchment']=pd.qcut(gdf_bng['sessions_catchment_5km'],q=3,labels=['low','mid','high'])

comparison=gdf_bng[['borough','s_bin','s_bin_catchment']]
flipped=comparison[comparison['s_bin'] != comparison['s_bin_catchment']]
print(f"\n{len(flipped)} / 32 boroughs change supply tier under catchment measure:")
print(flipped)

gdf_bng[['borough','sessions_catchment_2km','sessions_catchment_5km','s_bin_catchment']].to_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_catchment.csv",index=False)
print("\nSaved to GAP_data\\gap_scores_catchment.csv")

In [16]:
!pip install openpyxl

In [ ]:
#using centroids population based
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

centroids=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\LSOA_PopCentroids_EW_2021_V4_-5180599451571443473.csv")
centroids=centroids[['LSOA21CD','x','y']]
print("Centroids:",centroids.shape)
pop=pd.read_excel(r"C:\Users\Hp\Downloads\Project 2026 DS\sapelsoasyoa20222024.xlsx",sheet_name='Mid-2024 LSOA 2021',header=3  )
pop=pop[['LAD 2023 Code','LAD 2023 Name','LSOA 2021 Code','Total']]
pop=pop.rename(columns={'LSOA 2021 Code': 'LSOA21CD','LAD 2023 Name': 'borough','Total': 'population'})
print("Population:",pop.shape)
print(pop.head())

london_boroughs=[
    "Barking and Dagenham","Barnet","Bexley","Brent","Bromley","Camden","City of London","Croydon","Ealing","Enfield","Greenwich","Hackney","Hammersmith and Fulham","Haringey","Harrow","Havering","Hillingdon",
    "Hounslow","Islington","Kensington and Chelsea","Kingston upon Thames","Lambeth","Lewisham","Merton","Newham","Redbridge","Richmond upon Thames","Southwark","Sutton","Tower Hamlets","Waltham Forest","Wandsworth","Westminster"]
pop_london=pop[pop['borough'].isin(london_boroughs)].copy()
print(f"\nLondon LSOA rows: {len(pop_london)}")
print(pop_london['borough'].nunique(),"boroughs found")

lsoa_merged=pop_london.merge(centroids,on='LSOA21CD',how='inner')
print(f"\nMerged  {len(lsoa_merged)} rows")

def weighted_centroid(group):
    x=(group['x'] * group['population']).sum() / group['population'].sum()
    y=(group['y'] * group['population']).sum() / group['population'].sum()
    return pd.Series({'x_pw': x,'y_pw': y})

borough_pw_centroids=lsoa_merged.groupby('borough').apply(weighted_centroid).reset_index()
print("\npopulation weight borough centroids:")
print(borough_pw_centroids)

borough_pw_gdf=gpd.GeoDataFrame(borough_pw_centroids,geometry=[Point(xy) for xy in zip(borough_pw_centroids['x_pw'],borough_pw_centroids['y_pw'])],crs="EPSG:27700")

borough_pw_gdf.to_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\borough_population_weighted_centroids.csv",index=False)
print("\nsaved population weight centroids.")

In [ ]:
venues=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\OpenActive_Merged.csv")
venues_geo=venues.dropna(subset=['latitude','longitude']).copy()
venues_gdf=gpd.GeoDataFrame(venues_geo,geometry=gpd.points_from_xy(venues_geo['longitude'],venues_geo['latitude']),crs="EPSG:4326")
venues_gdf=venues_gdf.to_crs(epsg=27700)

buffer_distances={'sessions_catchment_pw_2km': 2000,'sessions_catchment_pw_5km': 5000}

for col_name,dist in buffer_distances.items():
    counts=[]
    for centroid in borough_pw_gdf.geometry:
        buffer=centroid.buffer(dist)
        count=venues_gdf[venues_gdf.geometry.within(buffer)].shape[0]
        counts.append(count)
    borough_pw_gdf[col_name]=counts

print(borough_pw_gdf[['borough','sessions_catchment_pw_2km','sessions_catchment_pw_5km']].sort_values('sessions_catchment_pw_5km',ascending=False))

# checking original with catechment
gap_equity=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_equity.csv")
compare=borough_pw_gdf[['borough','sessions_catchment_pw_5km']].merge(gap_equity[['borough','s_bin','sessions']],on='borough',how='left')
compare['s_bin_pw_catchment']=pd.qcut(compare['sessions_catchment_pw_5km'],q=3,labels=['low','mid','high'])

flipped_pw=compare[compare['s_bin'] != compare['s_bin_pw_catchment']]
print(f"\n{len(flipped_pw)} / 33 boroughs change category under population catchment:")
print(flipped_pw[['borough','s_bin','s_bin_pw_catchment']])

compare.to_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_pw_catchment.csv",index=False)
print("\nSaved to GAP_data\\gap_scores_pw_catchment.csv")

In [ ]:
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

panel=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\gapscore_v2.csv")
gap_equity=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_equity.csv")
panel=panel.merge(gap_equity[['borough','s_bin']],on='borough',how='left')

panel['post_covid']=panel['covid_affected'].astype(int)
panel['good_coverage']=(panel['s_bin'] == 'high').astype(int)
panel_did=panel[panel['borough'] != 'City of London'].copy()
print("Rows in DiD panel:",len(panel_did))
print(panel_did[['survey_year','covid_affected']].drop_duplicates())

#did??
pretrend=panel_did[panel_did['covid_affected'] == False]
pretrend_avg=pretrend.groupby(['survey_year','good_coverage'])['pct_inactive'].mean().reset_index()

fig,ax=plt.subplots(figsize=(8,5))
for coverage_val,label in [(1,'High coverage'),(0,'Low/mid coverage')]:
    sub=pretrend_avg[pretrend_avg['good_coverage'] == coverage_val]
    ax.plot(sub['survey_year'],sub['pct_inactive'],marker='o',label=label)
ax.set_title("Pre-COVID trends: high vs low/mid OpenActive coverage boroughs")
ax.set_ylabel("Avg. pct_inactive")
ax.set_xlabel("Survey year")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\parallel_trends_check.png")
plt.show()

print("\nPre-COVID trend values:")
print(pretrend_avg)

#fitting the model
did_model=smf.ols("pct_inactive ~ post_covid * good_coverage + C(borough) + C(survey_year)",data=panel_did).fit(cov_type='cluster',cov_kwds={'groups': panel_did['borough']})
print(did_model.summary())
interaction_coef=did_model.params.get('post_covid:good_coverage',None)
interaction_pval=did_model.pvalues.get('post_covid:good_coverage',None)
print(f"\nDiD coefficient: {interaction_coef:.4f}")
print(f"p value {interaction_pval:.4f}")

if interaction_pval is not None and interaction_pval < 0.05:
    direction="smaller" if interaction_coef < 0 else "larger"
    print(f"{direction} statistically significant")
else:
    print("No statistically significant ")

In [ ]:
#lesenibg parameters
did_model_simple=smf.ols("pct_inactive ~ post_covid * good_coverage + C(survey_year)",data=panel_did).fit(cov_type='cluster',cov_kwds={'groups': panel_did['borough']})
print(did_model_simple.summary())

In [ ]:
#weighting score

df=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\individual_level_london.csv")
london_avg_opportunity=df['readiness_opportunity'].mean()
london_avg_ability=df['readiness_ability'].mean()

borough_readiness=df.groupby('borough').agg(avg_opportunity=('readiness_opportunity','mean'),avg_ability=('readiness_ability','mean')).reset_index()
borough_readiness['opportunity_gap']=london_avg_opportunity - borough_readiness['avg_opportunity']
borough_readiness['ability_gap']=london_avg_ability - borough_readiness['avg_ability']
borough_readiness['dominant_barrier']=borough_readiness.apply(lambda r: 'opportunity' if r['opportunity_gap'] > r['ability_gap'] else 'ability',axis=1)
print(borough_readiness.sort_values('opportunity_gap',ascending=False))

In [ ]:
opportunity_barrier_activities=['walking','running','park','outdoor','community','informal']
ability_barrier_activities=['gym','fitness class','swimming','yoga','pilates' ]
def tag_activity(activity_type):
    a=str(activity_type).lower()
    if any(k in a for k in opportunity_barrier_activities):
        return 'opportunity'
    elif any(k in a for k in ability_barrier_activities):
        return 'ability'
    return 'general'

venues=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\OpenActive_Merged.csv")
venues['barrier_tag']=venues['activity_type'].apply(tag_activity)

In [ ]:
gap_equity=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_equity.csv")
#intervention
target_boroughs=gap_equity[gap_equity['class'].isin(['genuine desert','blind spot','blind spot risk'])]['borough']

recommendations=[]
for borough in target_boroughs:
    barrier=borough_readiness.loc[borough_readiness['borough'] == borough,'dominant_barrier'].values[0]
    local_venues=venues[venues['borough'] == borough]
    supply_by_tag=local_venues['barrier_tag'].value_counts(normalize=True)
    matching_supply_pct=supply_by_tag.get(barrier,0)
    score=1.0 - matching_supply_pct
    recommendations.append({
        'borough': borough,
        'class': gap_equity.loc[gap_equity['borough'] == borough,'class'].values[0],
        'dominant_barrier': barrier,
        'current_matching_supply_pct': round(matching_supply_pct * 100,1),
        'recommendation': (f"Prioritise low-{barrier}-barrier activity types "f"(e.g. {'informal outdoor/community sessions' if barrier=='opportunity' else 'low-cost structured fitness options'})"),
        'priority_score': round(score,3)})

rec_df=pd.DataFrame(recommendations).sort_values('priority_score',ascending=False)
print(rec_df)
rec_df.to_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\intervention_recommendations.csv",index=False)

under_monitored=gap_equity[gap_equity['class'] == 'under-monitored'][['borough','inactive','demand_score']]
print("\nUnder monitored boroughs")
print(under_monitored)
under_monitored.to_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\under_monitored_flags.csv",index=False)

In [ ]:
def generate_borough_summary(borough_row):
    return ( f"{borough_row['borough']} is classified as '{borough_row['class']}'. " f"Its inactivity rate is {borough_row['inactive']:.1f}%, "
        f"{'above' if borough_row['inactive'] > df['inactive'].mean() else 'below'} the London average. "f"Equity gap: {borough_row['equity_gap']:.1f} "f"({'a significant concern' if borough_row['equity_flag']=='high gap' else 'not a major concern'}).")

In [25]:
print(gap_equity['class'].value_counts())
print(target_boroughs.tolist())

class
average                        4
under-monitored                4
emerging desert                4
low need, low supply           4
well-served                    4
well-served (moderate need)    3
blind spot risk                3
adequately served              2
genuine desert                 2
blind spot                     2
Name: count, dtype: int64
['Bromley', 'Enfield', 'Greenwich', 'Hammersmith and Fulham', 'Lambeth', 'Redbridge', 'Richmond upon Thames']
